In [2]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline 

In [4]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [8]:
# build the voacabulary of characters and mapping to/from integers 
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [16]:
block_size = 3  # context length: how many characters do we take to predict the next one
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]   # crop and append 
X = torch.tensor(X)
Y = torch.tensor(Y)



emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [36]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [18]:
C = torch.randn((27,2))

In [20]:
C[5]

tensor([ 0.6791, -0.8449])

In [22]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([ 0.6791, -0.8449])

In [26]:
C[X].shape

torch.Size([32, 3, 2])

In [28]:
X[13,2]

tensor(1)

In [30]:
C[X][13,2]

tensor([ 0.3605, -0.4818])

In [32]:
C[1]

tensor([ 0.3605, -0.4818])

In [40]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [42]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [44]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [54]:
h = torch.tanh(emb.view(32,6) @ W1 +b1)
h.shape


torch.Size([32, 100])

In [58]:
h

tensor([[ 0.4498, -0.5831, -0.9785,  ..., -0.9973, -0.4216,  1.0000],
        [-0.1395,  0.9948,  0.9417,  ...,  0.5678,  0.7869,  0.8313],
        [-0.9999,  0.9999, -0.0604,  ..., -0.6105, -0.9991,  0.9997],
        ...,
        [ 0.9983, -0.1895,  0.8861,  ...,  0.8466, -0.9997, -0.9200],
        [-0.9287,  0.7782,  0.2750,  ...,  0.9403, -0.9990,  0.9909],
        [ 0.9984, -0.9999,  0.9587,  ...,  0.9995, -0.9999, -0.2437]])

In [62]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [64]:
logits = h @ W2 + b2

In [70]:
logits.shape

torch.Size([32, 27])

In [72]:
counts = logits.exp()

In [74]:
prob = counts / counts.sum(1, keepdim=True)

In [76]:
prob.shape

torch.Size([32, 27])

In [78]:
#--------more clean-------

In [80]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [98]:
g = torch.Generator().manual_seed(2147483647) 
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [100]:
sum(p.nelement() for p in parameters) # no of parameters

3481

In [110]:
for p in parameters:
    p.requires_grad = True

In [114]:
for _ in range(1000):

    # forward pass 
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 +b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    
    
    # backward pass 
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # update 
    for p in parameters:
        p.data += -0.1 * p.grad 
print(loss.item())

0.25608447194099426


In [92]:
prob.shape

torch.Size([32, 27])